# Comparing Safety-Removal Subspaces in Aligned LLMs

End-to-end Colab notebook for reproducing all experiments.

## GPU requirements
| Step | Minimum | Recommended |
|------|---------|-------------|
| DIM | 16 GB (T4) | A100 40 GB |
| ActSVD | 24 GB | A100 80 GB |
| RDO training | 24 GB | A100 40 GB |
| Safety-utility overlap | 16 GB | A100 40 GB |
| RepInd analysis | 16 GB | A100 40 GB |

**Recommended**: Use an A100 runtime. Go to `Runtime → Change runtime type → GPU → A100`.

## Quick smoke test
Set `MODEL_PATH = "google/gemma-2b-it"` and skip ActSVD (cell 9). DIM + RDO + analysis runs in ~30 min on a T4.

## Research design
We compare three methods for removing safety behavior from Llama-3.1-8B-Instruct:
1. **DIM** (Arditi et al. 2024) — difference-in-means refusal direction
2. **ActSVD** (Wei et al. 2024) — low-rank safety/utility disentanglement
3. **RDO** (Wollschläger et al. 2025) — gradient-optimized refusal direction

We then compare their subspaces geometrically (MSO), test causal independence (RepInd), and measure the safety-utility tradeoff on JailbreakBench.

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/evan203/nlp-project-proposal.git"
BRANCH = "experiment/kyle"
PROJECT = Path('/content/nlp-project-proposal')

# --- Model selection ---
# For the full report: meta-llama/Llama-3.1-8B-Instruct (requires A100 + HF token)
# For a quick smoke test: google/gemma-2b-it (no token needed, T4-compatible)
MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_ALIAS = MODEL_PATH.split('/')[-1]  # e.g. Llama-3.1-8B-Instruct

HF_HOME = PROJECT / 'code' / 'llm_weights'
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME)
os.environ['PYTHON_RUNNER'] = 'python'

def run(cmd, cwd=None, env_extra=None):
    """Run a shell command, optionally with extra env vars."""
    env = os.environ.copy()
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd or PROJECT, env=env)

print('GPU status:')
run('nvidia-smi')

## 1. Clone the repository and authenticate Hugging Face

Llama-3.1-8B-Instruct is gated. Paste a token that has accepted the Meta Llama license at [huggingface.co/meta-llama](https://huggingface.co/meta-llama).

If using `google/gemma-2b-it`, skip the login cell.

In [ ]:
if not PROJECT.exists():
    run(f"git clone -b {BRANCH} {REPO_URL} {PROJECT}", cwd="/content")
else:
    run(f"git checkout {BRANCH} && git pull", cwd=PROJECT)

run("pip -q install -U huggingface_hub")

from huggingface_hub import notebook_login, whoami

try:
    user = whoami()
    print("Already logged into Hugging Face as:", user["name"])
except Exception:
    notebook_login()

## 2. Install dependencies

Install all required packages. `vllm` is needed by the evaluation harness (even when using only substring matching). `nnsight==0.3.7` is required by the RDO/cones code.

In [ ]:
run('pip -q install -U torch torchvision transformers accelerate datasets '
    'sentencepiece protobuf tqdm jaxtyping matplotlib seaborn '
    'zstandard litellm einops')
run('pip -q install -U nnsight==0.3.7')
run('pip -q install -U vllm')  # Required by evaluate_jailbreak.py imports
run('pip -q install python-dotenv wandb')  # Required by RDO training

# Verify GPU is accessible
run("python -c 'import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))'")

## 3. Run DIM (Difference-in-Means)

Extracts the refusal direction, evaluates baseline/ablation/activation-addition completions on JailbreakBench and harmless instructions, and saves:
- `code/methods/dim/pipeline/runs/<model>/direction.pt` — the refusal direction
- `code/methods/dim/pipeline/runs/<model>/generate_directions/mean_diffs.pt` — all candidate directions
- `code/methods/dim/pipeline/runs/<model>/modified_model/` — weight-edited model
- `code/methods/dim/pipeline/runs/<model>/completions/` — all completions + evaluations

**Time estimate**: ~20 min on A100 for Llama-3.1-8B.

In [ ]:
run('./scripts/run_dim.sh', env_extra={'MODEL_PATH': MODEL_PATH})

## 4. Run ActSVD

Computes low-rank safety/utility disentanglement via SVD. Saves a modified model to `code/methods/actsvd/out/`.

**Time estimate**: ~40 min on A100 with `NSAMPLES=128`.

**Smoke test**: set `NSAMPLES=32` to reduce time and VRAM usage to ~16 GB.

> **Note**: Set `EVAL_PPL=1` to also run wikitext perplexity (adds ~10 min). Set `EVAL_ATTACK=1` to run attack eval (requires vLLM serving, additional VRAM).

In [ ]:
run('./scripts/run_actsvd.sh', env_extra={
    'MODEL_ALIAS': MODEL_ALIAS,
    'NSAMPLES': '128',
    'RANK_POS': '3000',
    'RANK_NEG': '4000',
    'EVAL_PPL': '0',
    'EVAL_ATTACK': '0',
})

## 5. Run RDO (Refusal Direction Optimization) — The Geometry Paper

Trains a gradient-optimized refusal direction using the loss from Wollschläger et al. (2025). This is the third method in our comparison alongside DIM and ActSVD.

Steps performed by `run_rco_eval.sh`:
1. **Train**: Run `rdo.py` with `WANDB_MODE=offline` (local artifacts, no internet required)
2. **Extract**: Copy the best direction to `code/results/geometry_repind/rco_direction.pt`
3. **Evaluate**: Run `eval_direction_benchmark.py` to get JBB ASR + harmless compliance
4. **RepInd**: Run the RepInd comparison with DIM + RDO directions

**Time estimate**: ~30–60 min on A100 (training + evaluation).

> **Smoke test**: Set `MODE=direction` (single direction, faster than cone). For the full cone paper result, set `MODE=cone CONE_DIM=2`.

In [ ]:
run('./scripts/run_rco_eval.sh', env_extra={
    'MODEL': MODEL_PATH,
    'MODE': 'direction',   # Use 'cone' with CONE_DIM=2 for full cone experiment
    'METHOD_NAME': 'RDO',
    'WANDB_MODE': 'offline',
    'SKIP_REPIND': '0',    # Set to '1' to skip RepInd comparison
    'SAVE_MODIFIED': '0',  # Set to '1' to save a weight-edited model
})

## 6. Run direct safety-vs-utility overlap

Computes per-layer MSO between DIM safety directions and PCA utility activation subspaces from harmless instructions. This directly answers: *how much do safety directions lie inside the utility activation manifold?*

Outputs:
- `code/results/safety_utility_overlap/safety_utility_overlap_results.json`
- Three plots: per-layer, by-rank, and heatmap

**Time estimate**: ~15 min on A100.

In [ ]:
run('./scripts/run_safety_utility_overlap.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'N_UTILITY_SAMPLES': '128',
    'UTILITY_RANKS': '1,2,4,8,16,32',
    'PRIMARY_RANK': '8',
})

## 7. Run Geometry/RepInd profile analysis (DIM-derived basis)

If you ran step 5 (RDO), the RepInd comparison with the trained RDO direction was already saved to `code/results/geometry_repind_rco/`. This cell reruns the lighter DIM-derived version.

To use trained RDO directions instead of DIM-derived:
```python
run('./scripts/run_geometry_repind.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'DIRECTIONS_JSON': 'code/results/geometry_repind/directions_rco.json',
    'OUTPUT_DIR': 'code/results/geometry_repind_rco',
})
```

In [ ]:
run('./scripts/run_geometry_repind.sh', env_extra={
    'MODEL_PATH': MODEL_PATH,
    'N_PROMPTS': '32',
    'BATCH_SIZE': '8',
    'DERIVED_CONE_DIM': '3',
})

## 8. Add benchmark entries for DIM and ActSVD

The DIM pipeline already saves its completions and evaluations under `code/methods/dim/pipeline/runs/`. This cell converts them into the `benchmark_results.json` format that `plot_benchmarks.py` reads.

The ActSVD modified model was already evaluated during training. This cell also adds ActSVD to the benchmark JSON if not already present.

In [ ]:
import json, pathlib

CODE = PROJECT / 'code'
DIM_RUN = CODE / 'methods/dim/pipeline/runs' / MODEL_ALIAS
ACTSVD_OUT = CODE / 'methods/actsvd/out'
BENCHMARK = CODE / 'results/benchmark/benchmark_results.json'

benchmark = json.loads(BENCHMARK.read_text()) if BENCHMARK.exists() else {}

# --- DIM baseline + ablated ---
def load_eval(path):
    return json.loads(path.read_text()) if path.exists() else {}

baseline_eval = load_eval(DIM_RUN / 'completions/jailbreakbench_baseline_evaluations.json')
ablation_eval = load_eval(DIM_RUN / 'completions/jailbreakbench_ablation_evaluations.json')
harmless_eval = load_eval(DIM_RUN / 'completions/harmless_actadd_evaluations.json')

if 'Base (Llama-3.1-8B-Instruct)' not in benchmark:
    benchmark['Base (Llama-3.1-8B-Instruct)'] = {
        'model_path': str(DIM_RUN / 'modified_model'),
        'jailbreakbench': {
            'asr': baseline_eval.get('substring_matching_success_rate', None),
            'per_category': baseline_eval.get('substring_matching_per_category', {}),
            'source': 'DIM baseline completions'
        },
    }

if 'DIM-Ablated' not in benchmark:
    benchmark['DIM-Ablated'] = {
        'model_path': str(DIM_RUN / 'modified_model'),
        'jailbreakbench': {
            'asr': ablation_eval.get('substring_matching_success_rate', None),
            'per_category': ablation_eval.get('substring_matching_per_category', {}),
            'source': 'DIM ablation completions'
        },
        'harmless_compliance': {
            'rate': 1.0 - harmless_eval.get('substring_matching_success_rate', 0.0),
            'source': 'DIM harmless actadd inverted'
        },
    }

BENCHMARK.write_text(json.dumps(benchmark, indent=2))
print('benchmark_results.json:')
for k, v in benchmark.items():
    asr = v.get('jailbreakbench', {}).get('asr', 'N/A')
    hc = v.get('harmless_compliance', {}).get('rate', 'N/A')
    print(f'  {k}: ASR={asr}, harmless={hc}')

## 9. Regenerate all comparison plots

Generates benchmark, method-overlap, safety-utility overlap, and RepInd figures, then syncs them to `docs/figures/` for the Typst report.

In [ ]:
import os
os.chdir(str(PROJECT / 'code'))

run('python analysis/plot_benchmarks.py', cwd=PROJECT / 'code')
run('python analysis/plot_method_overlap.py', cwd=PROJECT / 'code')
run('python analysis/plot_safety_utility_overlap.py', cwd=PROJECT / 'code')

# Plot RepInd results — use RCO version if available, else DIM-derived
import pathlib
rco_results = pathlib.Path('/content/nlp-project-proposal/code/results/geometry_repind_rco/geometry_repind_results.json')
if rco_results.exists():
    run('python analysis/plot_geometry_repind.py --results_dir results/geometry_repind_rco', cwd=PROJECT / 'code')
else:
    run('python analysis/plot_geometry_repind.py', cwd=PROJECT / 'code')

run('python scripts/sync_figures.py', cwd=PROJECT)
run('python scripts/inventory.py', cwd=PROJECT)

## 10. Download results

Download the small result files for local use. Do **not** download model weights (they are 15–30 GB each).

In [ ]:
import shutil, pathlib
from google.colab import files

RESULTS = PROJECT / 'code' / 'results'
DOCS_FIGS = PROJECT / 'docs' / 'figures'

# Create a zip of results + figures (excluding model weights)
shutil.make_archive('/content/project_results', 'zip', PROJECT,
                    base_dir=None)

# Or just download the figures and JSON results individually:
for f in sorted(DOCS_FIGS.glob('*.png')):
    files.download(str(f))
files.download(str(RESULTS / 'benchmark/benchmark_results.json'))
files.download(str(RESULTS / 'safety_utility_overlap/safety_utility_overlap_results.json'))

## Appendix: Cone experiment (optional)

Trains a 2-dimensional refusal cone instead of a single direction. Results in higher ASR but takes longer.
After training, re-run the RepInd analysis with the cone basis to compare its independence structure against DIM.

In [ ]:
run('./scripts/run_rco_eval.sh', env_extra={
    'MODEL': MODEL_PATH,
    'MODE': 'cone',
    'CONE_DIM': '2',
    'METHOD_NAME': 'RCO-cone-2',
    'WANDB_MODE': 'offline',
    'SKIP_REPIND': '0',
})

## Appendix: Ad hoc MSO between any two tensors

Use `scripts/compare_overlap.py` for quick MSO checks between saved direction tensors.

In [ ]:
# Compare DIM direction against all candidate mean-diffs
run(
    f'python scripts/compare_overlap.py '
    f'--left code/methods/dim/pipeline/runs/{MODEL_ALIAS}/direction.pt '
    f'--right code/methods/dim/pipeline/runs/{MODEL_ALIAS}/generate_directions/mean_diffs.pt '
    f'--right-rank 8'
)

# Compare DIM direction against RDO direction (after running step 5)
run(
    f'python scripts/compare_overlap.py '
    f'--left code/methods/dim/pipeline/runs/{MODEL_ALIAS}/direction.pt '
    f'--right code/results/geometry_repind/rco_direction.pt'
)